# Bokeh — Interactive Web Visualizations with Python Backends

## What is Bokeh?

Bokeh is a Python library for creating **interactive visualizations for the web**. It renders charts as HTML+JavaScript that run in any browser. What makes Bokeh unique is its support for **server-side Python callbacks** — the visualization can update in real-time based on Python logic, enabling true streaming dashboards.

**Real-world analogy**: If Plotly is a smart TV (great interactive content, limited customization of the underlying system), Bokeh is a full smart home system — you can connect Python code that runs on the server and update the visuals based on real-time data, user inputs, or database queries.

## Bokeh vs Plotly — Key Differences

| Feature | Bokeh | Plotly |
|---------|-------|--------|
| Standalone HTML | ✓ | ✓ |
| Python server callbacks | ✓ (Bokeh Server) | Limited (Dash) |
| Streaming/real-time data | ✓ (excellent) | Limited |
| Ease of use | Moderate | Easy (px API) |
| 3D plots | ✗ | ✓ |
| Jupyter support | ✓ | ✓ |
| Custom JS callbacks | ✓ (CustomJS) | Limited |

**Use Bokeh when**: Real-time streaming data, server-side interactivity, custom JavaScript widgets, or highly customized layouts.

## Prerequisites
- Python basics
- NumPy/Pandas

## Table of Contents
1. Installation & Setup
2. Core Concepts: Figure, Glyphs, Data Sources
3. Basic Plot Types (Line, Scatter, Bar, Patch)
4. ColumnDataSource: Bokeh's Data Container
5. Hover Tools & Interactivity
6. Widgets (Slider, Select, Button)
7. Layouts (Row, Column, Grid)
8. Linked Plots & Brushing
9. Exporting (HTML, PNG)
10. Common Pitfalls
11. Mini Project: Real-Time Stock Simulator
12. Interview Q&A
13. Resources

---

**Official Docs**: https://docs.bokeh.org/  
**Gallery**: https://docs.bokeh.org/en/latest/docs/gallery.html  
**YouTube (Bokeh tutorials)**: https://www.youtube.com/c/BokehViz  
**Tutorial**: https://docs.bokeh.org/en/latest/docs/first_steps.html

## 1. Installation & Setup

```bash
pip install bokeh pandas numpy
```

For Jupyter notebooks, call `output_notebook()` to display plots inline. For scripts, call `output_file('chart.html')` to save to HTML.

In [ ]:
from bokeh.plotting import figure, show, output_notebook, output_file, save
from bokeh.models import (ColumnDataSource, HoverTool, Slider, Select, 
                           Button, ColorBar, LinearColorMapper,
                           Range1d, Span, BoxAnnotation, Label)
from bokeh.layouts import row, column, gridplot
from bokeh.transform import factor_cmap, linear_cmap
from bokeh.palettes import Category10, Viridis256, RdYlGn11
import pandas as pd
import numpy as np
import bokeh

# For Jupyter: display plots inline
output_notebook()

print(f"Bokeh version: {bokeh.__version__}")

---
## 2. Core Concepts: Figure, Glyphs, Data Sources

### Building Blocks

```
Figure  — The canvas (axes, grid, toolbar)
  │
  └── Glyphs — Visual shapes: line, circle, rect, patch, wedge, quad...
       │
       └── Connected to a ColumnDataSource (Bokeh's DataFrame-like data container)
```

**Analogy**: The `Figure` is the empty stage. `Glyphs` are the actors. `ColumnDataSource` is the script — it tells each actor what to do and can be updated to make them respond dynamically.

### The Bokeh Workflow
1. Create a `figure()` — sets up the canvas with title, axes, toolbar
2. Add `glyphs` to the figure — `p.line()`, `p.circle()`, `p.rect()`, etc.
3. (Optional) Add `tools` — hover, zoom, pan, tap
4. `show(p)` — display in notebook or browser

In [ ]:
# ── Simplest Bokeh plot ────────────────────────────────────────────────────────
x = [1, 2, 3, 4, 5]
y = [6, 7, 2, 4, 5]

# Create a figure
p = figure(
    title='My First Bokeh Plot',
    x_axis_label='X Values',
    y_axis_label='Y Values',
    width=500, height=300,
    toolbar_location='above'  # 'above', 'below', 'left', 'right', None
)

# Add glyphs (you can add multiple to the same figure)
p.line(x, y, legend_label='Line', line_width=2, line_color='navy')
p.circle(x, y, legend_label='Points', size=10, color='orange', 
          fill_alpha=0.8)

# Add a horizontal reference line at y=5
hline = Span(location=5, dimension='width', line_color='red', 
              line_dash='dashed', line_width=1.5)
p.add_layout(hline)

# Style the legend
p.legend.location = 'top_right'
p.legend.click_policy = 'hide'  # Click legend to show/hide traces!

show(p)
print("▲ Try the toolbar: Zoom (scroll), Pan, Reset, Save (PNG)")
print("▲ Click legend items to toggle visibility!")

---
## 3. Basic Plot Types

| Glyph Method | Shape | Use For |
|-------------|-------|--------|
| `p.line()` | Line | Time series, trends |
| `p.circle()` | Circles | Scatter plots |
| `p.square()` | Squares | Scatter variation |
| `p.triangle()` | Triangles | Category markers |
| `p.vbar()` | Vertical bars | Bar charts |
| `p.hbar()` | Horizontal bars | Horizontal bar charts |
| `p.patch()` | Filled polygon | Area charts |
| `p.rect()` | Rectangle | Heatmaps, candlestick |
| `p.wedge()` / `p.annular_wedge()` | Pie slice | Pie/donut charts |
| `p.image()` / `p.image_rgba()` | Image | Heatmaps, raster data |

In [ ]:
np.random.seed(42)

# ── Time Series Line Chart ─────────────────────────────────────────────────────
days   = list(range(1, 91))  # 90 days
price  = 100 + np.cumsum(np.random.randn(90) * 2)
sma_20 = pd.Series(price).rolling(20).mean().values

p1 = figure(title='Stock Price with 20-day SMA', width=650, height=280,
             x_axis_label='Day', y_axis_label='Price ($)')

# Shaded area for uncertainty
p1.patch(days + days[::-1], 
          list(price + 5) + list(price - 5)[::-1],
          alpha=0.15, color='lightblue', legend_label='±5 band')
p1.line(days, price, line_width=2, color='steelblue', legend_label='Price')
p1.line(days, sma_20, line_width=2, color='orange', line_dash='dashed',
         legend_label='SMA-20')
p1.legend.location = 'top_left'
p1.legend.click_policy = 'hide'
show(p1)

# ── Vertical Bar Chart ─────────────────────────────────────────────────────────
categories = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
sales      = [120, 110, 130, 125, 180, 250, 220]
bar_colors = ['#e74c3c' if s < 130 else '#2ecc71' for s in sales]

p2 = figure(x_range=categories, title='Daily Sales', 
             width=500, height=280, x_axis_label='Day', y_axis_label='Sales')
p2.vbar(x=categories, top=sales, width=0.7, color=bar_colors, 
         line_color='white', legend_label='Sales')
p2.yaxis.axis_label = 'Units Sold'
show(p2)

# ── Scatter Plot with Multi-color ─────────────────────────────────────────────
from bokeh.sampledata.iris import flowers as iris_df

species_list = list(iris_df['species'].unique())
palette = Category10[len(species_list)]

source = ColumnDataSource(iris_df)

p3 = figure(title='Iris Dataset: Sepal Length vs Width', 
             width=550, height=300)
p3.circle('sepal_length', 'sepal_width', source=source,
           color=factor_cmap('species', palette, species_list),
           legend_field='species', size=8, alpha=0.7)
p3.legend.location = 'top_right'
p3.legend.click_policy = 'hide'
p3.xaxis.axis_label = 'Sepal Length (cm)'
p3.yaxis.axis_label = 'Sepal Width (cm)'
show(p3)

---
## 4. ColumnDataSource: Bokeh's Data Container

The `ColumnDataSource` (CDS) is central to Bokeh. It's a dictionary-like object that maps column names to arrays. You can:
- Use it in multiple plots to **link them** (selecting in one highlights in others)
- **Update it** to make plots respond to widgets without redrawing
- Use it with **streaming** to add data in real-time

**Analogy**: CDS is like a live spreadsheet. When you update a cell, all the charts that reference that spreadsheet automatically update.

In [ ]:
from bokeh.models import ColumnDataSource

# ── Create from dict ──────────────────────────────────────────────────────────
data = {
    'x': [1, 2, 3, 4, 5],
    'y': [5, 3, 8, 2, 7],
    'label': ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon'],
    'size':  [10, 15, 20, 12, 18],
    'color': ['red', 'blue', 'green', 'purple', 'orange']
}
source = ColumnDataSource(data=data)

print("CDS column names:", list(source.data.keys()))
print("Number of rows:",   len(source.data['x']))

# ── Create from DataFrame ──────────────────────────────────────────────────────
df = pd.DataFrame({'x': range(10), 'y': np.random.randn(10), 'category': ['A','B']*5})
source_df = ColumnDataSource(df)
print("\nFrom DataFrame:", list(source_df.data.keys()))

# ── Use CDS in a plot with full hover ─────────────────────────────────────────
hover = HoverTool(tooltips=[
    ('Label', '@label'),
    ('x',     '@x'),
    ('y',     '@y'),
])

p = figure(title='ColumnDataSource: Hover for Details', width=500, height=300,
            tools=[hover, 'pan', 'wheel_zoom', 'reset'])
p.circle('x', 'y', source=source, size='size', color='color', alpha=0.8)

# Add text labels next to each point
from bokeh.models import LabelSet
labels = LabelSet(x='x', y='y', text='label', source=source,
                   x_offset=5, y_offset=5, text_font_size='9pt', text_color='gray')
p.add_layout(labels)

show(p)
print("▲ Hover over circles to see labels and coordinates!")

---
## 5. Hover Tools & Advanced Interactivity

Bokeh's hover tool is highly customizable. You can show any column from your data, format values, and even show HTML with images.

**Hover tooltip format**:
- `@column_name` — display value from CDS column
- `@column_name{0.00}` — format with 2 decimal places
- `@column_name{0,0}` — thousands separator
- `@column_name{%0.1f}` — percentage format

In [ ]:
np.random.seed(0)

# ── Rich interactive scatter plot ──────────────────────────────────────────────
n = 100
data = {
    'company':   [f'Company {i}' for i in range(n)],
    'revenue':   np.random.uniform(5, 500, n),       # M$
    'employees': np.random.randint(50, 10000, n),
    'profit_margin': np.random.uniform(0.02, 0.35, n),
    'sector':    np.random.choice(['Tech', 'Finance', 'Healthcare', 'Retail'], n)
}
data['profit'] = data['revenue'] * data['profit_margin']
df_companies = pd.DataFrame(data)

source = ColumnDataSource(df_companies)

# Define hover tooltip
hover = HoverTool(tooltips=[
    ('Company',        '@company'),
    ('Sector',         '@sector'),
    ('Revenue',        '@revenue{$0.0f}M'),
    ('Employees',      '@employees{0,0}'),
    ('Profit Margin',  '@profit_margin{0.0%}'),
    ('Profit',         '@profit{$0.0f}M'),
])

sectors      = list(df_companies['sector'].unique())
palette      = Category10[len(sectors)]

p = figure(
    title='Company Analysis: Revenue vs Employees',
    width=700, height=400,
    x_axis_label='Revenue ($M)',
    y_axis_label='Number of Employees',
    tools=[hover, 'pan', 'wheel_zoom', 'box_select', 'reset'],
    tooltips=None  # We'll use the HoverTool we configured above
)
p.add_tools(hover)

# Size scaled by profit margin, color by sector
p.circle(
    x='revenue', y='employees',
    source=source,
    size=df_companies['profit_margin'] * 60,  # Size = profit margin
    color=factor_cmap('sector', palette, sectors),
    legend_field='sector',
    alpha=0.65,
    line_color='white'
)
p.legend.location = 'top_left'
p.legend.click_policy = 'hide'

# Add shaded region for high-revenue zone
box = BoxAnnotation(left=300, fill_alpha=0.05, fill_color='green',
                     line_color='green', line_dash='dashed')
p.add_layout(box)
label = Label(x=310, y=9000, text='High Revenue Zone', text_color='green',
               text_font_size='10pt')
p.add_layout(label)

show(p)
print("▲ Hover for details! Use Box Select tool to select groups. Click legend to filter.")

---
## 6. Layouts: Row, Column, Grid

Bokeh lets you arrange multiple plots into layouts:
- `row(p1, p2, p3)` — side by side
- `column(p1, p2, p3)` — stacked vertically
- `gridplot([[p1, p2], [p3, p4]])` — 2D grid with shared toolbar

In [ ]:
from bokeh.layouts import row, column, gridplot

np.random.seed(42)
x = np.linspace(0, 4*np.pi, 200)

# Create 4 plots
p1 = figure(title='Sine Wave', width=350, height=250)
p1.line(x, np.sin(x), line_width=2, color='navy')

p2 = figure(title='Cosine Wave', width=350, height=250)
p2.line(x, np.cos(x), line_width=2, color='crimson')

p3 = figure(title='Damped Sine', width=350, height=250)
p3.line(x, np.sin(x) * np.exp(-x/8), line_width=2, color='forestgreen')

p4 = figure(title='Sine × Cosine', width=350, height=250)
p4.line(x, np.sin(x) * np.cos(x), line_width=2, color='darkorange')

# Grid layout with shared toolbar
grid = gridplot([[p1, p2], [p3, p4]], 
                 merge_tools=True,  # Share one toolbar for all panels
                 toolbar_location='right')

show(grid)

---
## 7. Linked Plots & Brushing

**Linked brushing** is a powerful EDA technique: **selecting data in one plot highlights the same data points in all linked plots**. This is unique to Bokeh (Plotly needs Dash for this).

**How it works**: Share the same `ColumnDataSource` between multiple plots. When you select points in one, the selection state in the shared CDS triggers highlighting in all plots.

In [ ]:
from bokeh.models import BoxSelectTool, LassoSelectTool

np.random.seed(42)
n = 200

# One shared data source — this is the key to linked brushing!
shared_source = ColumnDataSource({
    'x':    np.random.normal(0, 1, n),
    'y':    np.random.normal(0, 1, n),
    'size': np.random.uniform(3, 15, n),
    'color_val': np.random.uniform(0, 1, n)
})

TOOLS = 'box_select,lasso_select,pan,wheel_zoom,reset'

# Left plot: X vs Y
left = figure(title='Left: Select here...', width=380, height=320,
               tools=TOOLS, x_axis_label='X', y_axis_label='Y')
left.circle('x', 'y', source=shared_source, size='size',
             color='navy', alpha=0.6,
             nonselection_alpha=0.1)  # Non-selected points dim to 10% opacity

# Right plot: X vs size — SAME SOURCE → linked!
right = figure(title='Right: ...selection appears here too!', width=380, height=320,
                tools=TOOLS, x_axis_label='X', y_axis_label='Size')
right.circle('x', 'size', source=shared_source, size='size',
              color='crimson', alpha=0.6,
              nonselection_alpha=0.1)

show(row(left, right))
print("▲ Use Box Select or Lasso in the LEFT plot.")
print("  The SAME points highlight in the RIGHT plot automatically!")
print("  This is 'linked brushing' — great for exploring multi-dimensional data.")

---
## 8. Exporting Bokeh Figures

In [ ]:
import os
from bokeh.plotting import output_file

# Create a sample plot
x = np.linspace(0, 10, 100)
p_export = figure(title='Export Example', width=600, height=300)
p_export.line(x, np.sin(x), line_width=2, color='navy')
p_export.circle(x[::10], np.sin(x[::10]), size=8, color='red')

# ── Export as interactive HTML ─────────────────────────────────────────────────
output_file('/tmp/bokeh_chart.html', title='Bokeh Export')
save(p_export)
print(f"HTML saved: {os.path.getsize('/tmp/bokeh_chart.html'):,} bytes")
print("Open in any browser — fully interactive with zoom/pan/hover!")

# ── Export as PNG (requires selenium + webdriver) ──────────────────────────────
try:
    from bokeh.io import export_png
    export_png(p_export, filename='/tmp/bokeh_chart.png')
    print(f"PNG saved: {os.path.getsize('/tmp/bokeh_chart.png'):,} bytes")
except Exception as e:
    print(f"PNG export needs: pip install selenium geckodriver-autoinstaller")
    print(f"(Error: {type(e).__name__})")

print("\nFor a Bokeh Server app with real-time updates:")
print("  bokeh serve app.py --show")
print("See: https://docs.bokeh.org/en/latest/docs/user_guide/server.html")

---
## 9. Common Pitfalls

| Pitfall | Problem | Fix |
|---------|---------|-----|
| Forgot `output_notebook()` | No output in Jupyter | Call `output_notebook()` at the top |
| `show()` opens new browser tab | Expected inline in Jupyter | Make sure `output_notebook()` was called |
| `output_file()` before `output_notebook()` | Switches back to file mode | Call `output_notebook()` AFTER `output_file()`, or only use one |
| `str` type for x-axis categories | Error with `vbar` | Wrap in `figure(x_range=['A','B','C'])` for categorical |
| Widgets don't work in static HTML | `on_change` callbacks need a server | Use Bokeh Server (`bokeh serve`) for server callbacks; use `CustomJS` for client-side |
| `ColumnDataSource` length mismatch | Error: all columns must have same length | Verify all arrays have equal length before creating CDS |
| Static PNG needs geckodriver | Import error | `pip install selenium` + install Firefox + geckodriver, OR use Plotly for static exports |

---
## 10. Mini Project: Interactive Financial Dashboard

**Scenario**: Build a comprehensive financial visualization with:
1. Candlestick-style OHLC price chart
2. Volume bars
3. Moving average lines
4. Hover tooltips with full OHLC data
5. Linked x-range (scroll price chart → volume chart moves together)

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, Span
from bokeh.layouts import column
import pandas as pd
import numpy as np

np.random.seed(2024)

# ── Simulate OHLCV data ────────────────────────────────────────────────────────
n_days = 120
dates  = pd.date_range('2024-01-01', periods=n_days, freq='B')  # Business days

# Simulate price movement
close  = 100 + np.cumsum(np.random.randn(n_days) * 2)
open_  = close + np.random.randn(n_days) * 1.5
high   = np.maximum(close, open_) + np.abs(np.random.randn(n_days)) * 1.5
low    = np.minimum(close, open_) - np.abs(np.random.randn(n_days)) * 1.5
volume = np.random.randint(1_000_000, 10_000_000, n_days)

df = pd.DataFrame({
    'date': dates, 'open': open_, 'high': high, 'low': low, 'close': close, 'volume': volume
})

# Moving averages
df['sma_20'] = df['close'].rolling(20).mean()
df['sma_50'] = df['close'].rolling(50).mean()

# Candle color: green if close > open, red otherwise
df['color'] = np.where(df['close'] >= df['open'], '#2ecc71', '#e74c3c')

# Date as string for categorical x-axis (simpler for candlestick)
df['date_str'] = df['date'].dt.strftime('%Y-%m-%d')

source = ColumnDataSource(df)

# ── Price Chart (OHLC as rectangle + wick) ────────────────────────────────────
hover_price = HoverTool(tooltips=[
    ('Date',   '@date_str'),
    ('Open',   '@open{$0.00}'),
    ('High',   '@high{$0.00}'),
    ('Low',    '@low{$0.00}'),
    ('Close',  '@close{$0.00}'),
    ('Volume', '@volume{0,0}'),
], names=['candle_bodies'])

p_price = figure(
    title='Stock Price — OHLC Chart with Moving Averages',
    x_range=(df['date_str'].iloc[0], df['date_str'].iloc[-1]),
    x_axis_type='auto',
    y_axis_label='Price ($)',
    width=750, height=380,
    tools=[hover_price, 'xpan', 'xwheel_zoom', 'reset', 'box_select']
)
# NOTE: For true date x-axis: use x_axis_type='datetime' and date as actual dates
# Here we use category strings for simplicity
# Using dates properly
p_price2 = figure(
    title='Stock Price — OHLC Chart with Moving Averages',
    x_axis_type='datetime',
    y_axis_label='Price ($)',
    width=750, height=380,
    tools=['xpan', 'xwheel_zoom', 'reset']
)

# Wicks (high-low lines)
p_price2.segment('date', 'high', 'date', 'low', source=source,
                   color='color', line_width=1)

# Candle bodies (open-close rectangles)
w = 12*60*60*1000  # 12 hours in milliseconds (bar width)
p_price2.vbar(x='date', top='open', bottom='close', source=source,
               width=w, color='color', line_color='white', line_width=0.5,
               name='candle_bodies')

# Moving averages
p_price2.line('date', 'sma_20', source=source, line_width=2, 
               color='orange', alpha=0.8, legend_label='SMA-20')
p_price2.line('date', 'sma_50', source=source, line_width=2, 
               color='purple', alpha=0.8, legend_label='SMA-50')

p_price2.add_tools(HoverTool(names=['candle_bodies'], tooltips=[
    ('Date',   '@date{%F}'),
    ('OHLC',  'O:@open{$0.0f} H:@high{$0.0f} L:@low{$0.0f} C:@close{$0.0f}'),
    ('Volume', '@volume{0,0}'),
], formatters={'@date': 'datetime'}))

p_price2.legend.location = 'top_left'
p_price2.legend.click_policy = 'hide'
p_price2.xgrid.grid_line_color = None

# ── Volume Chart (linked x-axis) ───────────────────────────────────────────────
p_vol = figure(
    title='Volume',
    x_axis_type='datetime',
    x_range=p_price2.x_range,   # LINK to price chart!
    y_axis_label='Volume',
    width=750, height=150,
    tools=['xpan', 'xwheel_zoom', 'reset']
)
p_vol.vbar(x='date', top='volume', source=source,
            width=w, color='color', alpha=0.7)
p_vol.yaxis.formatter.use_scientific = False

# ── Show linked layout ─────────────────────────────────────────────────────────
show(column(p_price2, p_vol))
print("▲ Zoom/Pan the price chart — the volume chart moves together!")
print("  Green candle = price went UP | Red = price went DOWN")
print("  This is 'linked ranges' — a Bokeh specialty.")

---
## 11. Interview Q&A

**Q1: What is Bokeh, and how does it differ from Plotly?**  
**A**: Both create interactive HTML visualizations. Key differences: Bokeh supports server-side Python callbacks via Bokeh Server (your Python code runs live and updates charts), while Plotly uses Dash for this. Bokeh is better for streaming real-time data and linked brushing across multiple plots. Plotly has a simpler API (Plotly Express) and better 3D support. For most use cases, either works.

---
**Q2: What is a ColumnDataSource and why is it important?**  
**A**: `ColumnDataSource` is Bokeh's primary data container — a dictionary mapping column names to equal-length arrays. It's important because: (1) multiple plots can share one CDS, enabling linked selection/brushing; (2) updating the CDS's data dict updates all connected plots live; (3) it enables streaming (appending data) for real-time dashboards.

---
**Q3: What is linked brushing and how does Bokeh enable it?**  
**A**: Linked brushing means selecting data points in one plot automatically highlights the same data points in all linked plots. Bokeh enables this by having multiple plots share the same `ColumnDataSource`. When you select points in one plot, the selection state is stored in the shared CDS, and all other plots referencing it respond automatically.

---
**Q4: What is the difference between a standalone Bokeh HTML file and a Bokeh Server app?**  
**A**: A standalone HTML file (from `output_file` + `save`) is fully self-contained and works offline — but Python callbacks (using `widget.on_change()`) don't work because there's no Python running. A Bokeh Server app (`bokeh serve app.py`) keeps Python running, enabling live Python callbacks that can query databases, run ML models, or process streaming data.

---
**Q5: How do you create a linked x-axis between two plots (like price + volume)?**  
**A**: Pass the first plot's `x_range` to the second: `p2 = figure(x_range=p1.x_range, ...)`. Now zooming/panning p1 automatically moves p2's view to the same x window. This works for both linear and datetime axes.

---
## 12. Resources

### Official
- **Bokeh Documentation**: https://docs.bokeh.org/
- **First Steps Tutorial**: https://docs.bokeh.org/en/latest/docs/first_steps.html
- **Gallery**: https://docs.bokeh.org/en/latest/docs/gallery.html
- **Bokeh Server Guide**: https://docs.bokeh.org/en/latest/docs/user_guide/server.html

### YouTube
- **Bryan Van de Ven (Bokeh creator) PyCon talk**: https://www.youtube.com/watch?v=snbWOkxQCgo
- **Bokeh Interactive Tutorial**: https://www.youtube.com/watch?v=2TR_6VaVSOs

### Articles
- **Real Python — Bokeh Guide**: https://realpython.com/python-data-visualization-bokeh/
- **Bokeh vs Plotly comparison**: https://towardsdatascience.com/bokeh-vs-plotly

---
## Summary & What's Next

### What You Learned
| Concept | Key Point |
|---------|----------|
| Architecture | Figure → Glyphs → ColumnDataSource |
| Core glyphs | line, circle, vbar, hbar, rect, patch, segment |
| ColumnDataSource | Shared data container enabling linked selection |
| HoverTool | `@column_name{format}` for custom tooltips |
| Layouts | `row()`, `column()`, `gridplot()` |
| Linked brushing | Share CDS between plots → selection propagates |
| Linked axes | Share `x_range` between plots → zoom syncs |
| Export | `output_file()` + `save()` for standalone HTML |

### What's Next?
- **Next Notebook**: Altair — declarative Grammar-of-Graphics visualization
- **Practice**: Build a Bokeh Server app with a date range slider that updates a stock chart
- **Challenge**: Use Bokeh streaming to create a live updating sine wave (use `ColumnDataSource.stream()`)

> **Key insight**: Bokeh's superpower is linked plots and streaming data. When you need multiple charts that talk to each other, or real-time data pipelines, Bokeh is the right choice. For most dashboard use cases, Plotly+Dash vs Bokeh+Server is a matter of preference.